# 06G - Support Vector Machine (SVM) Classifier (Execution-Ready)

Optimized professional notebook. Run **Run All** to generate and save outputs.

In [ ]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay,
    PrecisionRecallDisplay
)

df = pd.read_csv("american_bankruptcy.csv")
df["target"] = df["status_label"].map({"alive":0,"failed":1})

display(df.head())
print(df.shape)


## Data Preparation

In [ ]:
drop_cols = ["status_label","target"]
if "company_name" in df.columns:
    drop_cols.append("company_name")

X = df.drop(columns=drop_cols)
y = df["target"]

num = X.select_dtypes(include="number").columns
cat = X.select_dtypes(exclude="number").columns

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("enc", OneHotEncoder(handle_unknown="ignore"))
    ]), cat)
])


## Train SVM Model

In [ ]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        probability=True,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)


## Cross Validation

In [ ]:
cv = cross_validate(
    model,
    X_train,
    y_train,
    cv=3,
    scoring=["accuracy","precision","recall","f1","roc_auc"]
)

display(pd.DataFrame(cv).describe())


## Model Evaluation

In [ ]:
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:,1]

metrics = pd.DataFrame({
    "Metric":["Accuracy","Precision","Recall","F1","ROC-AUC"],
    "Value":[
        accuracy_score(y_test,pred),
        precision_score(y_test,pred),
        recall_score(y_test,pred),
        f1_score(y_test,pred),
        roc_auc_score(y_test,prob)
    ]
})

display(metrics)
print(classification_report(y_test,pred))


## Visualizations

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test,pred)
plt.show()

RocCurveDisplay.from_predictions(y_test,prob)
plt.show()

PrecisionRecallDisplay.from_predictions(y_test,prob)
plt.show()


## Save Model

In [ ]:
joblib.dump(model,"svm_model.joblib")
print("Saved svm_model.joblib")


## Executive Summary

In [ ]:
print("- SVM is effective for complex decision boundaries.")
print("- Compare SVM performance with tree-based ensemble models.")
print("- Use the best-performing model for final selection and tuning.")
